[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iamjamesbowden/AG983/blob/main/class2/AG983_Week07_Workshop3_FacilitatorDebrief.ipynb)

## AG983 | Workshop 3 Facilitator Debrief

### Decision Audit and Results Analysis

*Dr James Bowden, Strathclyde Business School*

---

This notebook pulls the full results sheet from Google Sheets, validates the data, and generates structured debrief materials covering methodological decisions (Part 1) and substantive findings (Part 2). Run each cell in order before the debrief begins. The visualisations are designed to support discussion, not replace it -- the facilitator prompts in each cell indicate which patterns and divergences are worth drawing out.

**Prerequisites:** Google Sheet populated with student data; service account credentials JSON stored in Colab Secrets as `GSHEET_CREDS`.

In [ ]:
#@title Setup -- run this cell first
!pip install gspread google-auth pandas matplotlib seaborn wordcloud numpy -q

import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud
import gspread
from google.oauth2.service_account import Credentials
from IPython.display import display, Markdown
from google.colab import userdata

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})

print("Setup complete.")


In [ ]:
#@title Configuration -- set SPREADSHEET_ID before running

SPREADSHEET_ID = "1UrgwAXYkJabGAsfI4uxAdTEKkHE5deHUbk7yxvF9DBg"
WORKSHEET_NAME = "Sheet1"   # must match student notebook / Apps Script SHEET_NAME

FIRM_COLOURS = {
    "Boeing":        "#1f77b4",
    "Intel":         "#ff7f0e",
    "Peloton":       "#2ca02c",
    "SVB Financial": "#d62728",
}

# Column names matching the Google Sheets schema written by the student notebook.
# Source of truth: apps_script.js column order.
EXPECTED_COLS = [
    "student_id", "team_name", "firm", "year", "section", "normalisation",
    "stopwords", "remove_numbers", "lowercase", "dictionary",
    "net_sentiment_lm", "net_sentiment_hiv4", "fog_score", "readability_frame",
    "cosine_similarity", "tf_or_tfidf", "comparison_pair", "analyst_note",
    "timestamp"
]

print("Configuration loaded.")


In [ ]:
#@title Data pull -- fetch results from Google Sheets

def load_results():
    """Authenticate with gspread using Colab Secrets and return a clean DataFrame."""
    creds_json = userdata.get("GSHEET_CREDS")
    creds_dict = json.loads(creds_json)
    scopes = [
        "https://www.googleapis.com/auth/spreadsheets.readonly",
        "https://www.googleapis.com/auth/drive.readonly",
    ]
    creds = Credentials.from_service_account_info(creds_dict, scopes=scopes)
    gc    = gspread.authorize(creds)
    ws    = gc.open_by_key(SPREADSHEET_ID).worksheet(WORKSHEET_NAME)
    rows  = ws.get_all_records()
    df    = pd.DataFrame(rows)
    return df

df = load_results()

# Validate columns
missing_cols = [c for c in EXPECTED_COLS if c not in df.columns]
if missing_cols:
    print(f"WARNING: missing columns: {missing_cols}")

# Coerce numeric columns
for col in ["net_sentiment_lm", "net_sentiment_hiv4", "fog_score", "tf_or_tfidf", "cosine_similarity", "year"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# TEAM_LABELS: sorted by firm (FIRM_COLOURS order), then team name
firm_order = list(FIRM_COLOURS.keys())
df["_firm_rank"] = df["firm"].map({f: i for i, f in enumerate(firm_order)}).fillna(99)
df = df.sort_values(["_firm_rank", "team_name"]).reset_index(drop=True)
TEAM_LABELS = df["team_name"].tolist()

# Completeness check
print(f"Rows loaded: {len(df)}")
print(f"Teams: {sorted(df.team_name.unique())}")
print(f"Firms represented: {sorted(df.firm.unique())}")

incomplete = df[df[EXPECTED_COLS].isnull().any(axis=1)]
if len(incomplete):
    print(f"\nTeams with missing values ({len(incomplete)}):")
    print(incomplete[["team_name","firm"]].to_string(index=False))
else:
    print("\nAll rows complete -- no missing values.")

display(df[["team_name","firm","year","section","normalisation",
            "stopwords","dictionary","readability_frame",
            "net_sentiment_lm","fog_score","tf_or_tfidf","cosine_similarity"]].to_string(index=False))


---

## Part 1: Decision Audit

This section visualises the methodological choices each team made at Checkpoints 1 through 5. Use these outputs to identify divergences across teams working on the same firm and to structure the methodological discussion portion of the debrief.

### 1.1 Decision overview table

> "Start here. Ask teams to scan the table and identify any pair of teams working on the same firm who made different choices. These are the conversations worth having. Each divergence is a methodological question, not a mistake."

In [ ]:
#@title Cell 1.1 -- Decision overview table

decision_cols = [
    "team_name", "firm", "year", "section",
    "normalisation", "stopwords",
    "dictionary", "readability_frame", "comparison_pair"
]
summary = df[decision_cols].copy()

# Shorten long labels for display
summary["normalisation"] = summary["normalisation"].str.replace(
    "Lemmatisation (spaCy en_core_web_md)", "Lemmatisation", regex=False)
summary["readability_frame"] = summary["readability_frame"].str.replace(
    "Li (2008): complexity signals obfuscation", "Li (2008)", regex=False).str.replace(
    "Bushee et al. (2018): complexity reflects legitimate subject matter", "Bushee (2018)", regex=False).str.replace(
    "Not yet determined", "Undecided", regex=False)

# Rename columns for display
col_labels = {
    "team_name": "Team", "firm": "Firm", "year": "Year",
    "section": "Section", "normalisation": "Normalisation",
    "stopwords": "Stop words", "dictionary": "Dictionary",
    "readability_frame": "Readability frame", "comparison_pair": "Year pair"
}
display_df = summary.rename(columns=col_labels)

# Highlight rows where two teams share a firm
firm_counts = df["firm"].value_counts()
multi_firm = set(firm_counts[firm_counts > 1].index)

def highlight_multi(row):
    if row["Firm"] in multi_firm:
        return ["background-color: #fff9c4"] * len(row)
    return [""] * len(row)

styled = display_df.style.apply(highlight_multi, axis=1)
display(styled)
print("\nYellow rows: firms with multiple teams (most likely to show divergence).")


### 1.2 Pre-processing decision map

> "Pre-processing choices are upstream of everything else. A team that used the NLTK stop word list will have suppressed uncertainty markers before reaching the L&M dictionary. Ask: did any team notice in Checkpoint 3 that their Uncertainty score looked low? Could the stop word choice explain it? Which combination is most defensible for a published 10-K study, and why?"

In [ ]:
#@title Cell 1.2 -- Pre-processing decision map

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: heatmap of normalisation x stopwords
norm_order = ["None", "Porter Stemmer", "Lemmatisation (spaCy en_core_web_md)"]
sw_order   = ["None", "NLTK standard", "L&M-aware custom list"]

heat_data = pd.crosstab(
    pd.Categorical(df["normalisation"], categories=norm_order, ordered=True),
    pd.Categorical(df["stopwords"],     categories=sw_order,   ordered=True)
).reindex(index=norm_order, columns=sw_order, fill_value=0)

sns.heatmap(heat_data, annot=True, fmt="d", cmap="Blues", linewidths=0.5,
            ax=axes[0], cbar=False)
axes[0].set_title("Normalisation x Stop-word choice (team count)")
axes[0].set_xlabel("Stop-word list")
axes[0].set_ylabel("Normalisation")

# Right: stacked bar per firm
norm_short = {
    "None": "None",
    "Porter Stemmer": "Porter",
    "Lemmatisation (spaCy en_core_web_md)": "Lemma"
}
df["_norm_short"] = df["normalisation"].map(norm_short)
pivot = df.groupby(["firm","_norm_short"]).size().unstack(fill_value=0)
pivot = pivot.reindex(firm_order, fill_value=0)
palette_norm = {"None": "#aec7e8", "Porter": "#1f77b4", "Lemma": "#084594"}
for col_name in ["None", "Porter", "Lemma"]:
    if col_name in pivot.columns:
        pivot[col_name].plot(kind="bar", ax=axes[1],
                             color=palette_norm.get(col_name, "grey"),
                             label=col_name, stacked=True)
axes[1].set_title("Normalisation choice by firm")
axes[1].set_xlabel("")
axes[1].set_ylabel("Team count")
axes[1].legend(title="Normalisation", bbox_to_anchor=(1.01, 1))
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


### 1.3 Dictionary choice distribution

> "If any team chose Harvard IV-4 only, ask them: what did your net sentiment score look like, and how would it have changed under L&M? If the divergence panel was available to them, what specific words drove the difference? The Loughran and McDonald (2011) 'liability' example should be findable in almost any financial filing."

In [ ]:
#@title Cell 1.3 -- Dictionary choice distribution

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Left: overall pie
dict_counts = df["dictionary"].value_counts()
palette_dict = {
    "Harvard IV-4 only":    "#ff7f0e",
    "L&M only":             "#1f77b4",
    "Both (comparison mode)": "#2ca02c",
}
colours_pie = [palette_dict.get(k, "#cccccc") for k in dict_counts.index]
axes[0].pie(dict_counts.values, labels=dict_counts.index, colors=colours_pie,
            autopct="%1.0f%%", startangle=140, pctdistance=0.8)
axes[0].set_title("Dictionary choice (all teams)")

# Right: net sentiment by dictionary, for teams that ran both
both_df = df[df["dictionary"] == "Both (comparison mode)"].copy()
if len(both_df) > 0:
    both_df = both_df.sort_values("team_name")
    x = np.arange(len(both_df))
    w = 0.35
    bars_lm   = axes[1].bar(x - w/2, both_df["net_sentiment_lm"],   w, label="L&M",         color="#1f77b4")
    bars_harv = axes[1].bar(x + w/2, both_df["net_sentiment_hiv4"], w, label="Harvard IV-4", color="#ff7f0e")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(both_df["team_name"], rotation=30, ha="right")
    axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[1].set_title("L&M vs Harvard IV-4 net sentiment\n(teams using Both)")
    axes[1].set_ylabel("Net sentiment score")
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, "No teams selected Both\n(comparison mode)",
                 ha="center", va="center", transform=axes[1].transAxes, fontsize=12)
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()


### 1.4 Readability interpretation frame distribution

> "The 'Undecided' responses are worth discussing carefully -- they are often the most epistemically honest. Ask the teams who chose one of the two definitive frames: what specific evidence in your output pushed you toward that interpretation? Ask the 'Undecided' teams: what would you need to see to make a decision? These are the kinds of questions that appear in the examination."

In [ ]:
#@title Cell 1.4 -- Readability interpretation frame distribution

frame_map = {
    "Li (2008): complexity signals obfuscation":                           "Li (2008)",
    "Bushee et al. (2018): complexity reflects legitimate subject matter": "Bushee (2018)",
    "Not yet determined":                                                  "Undecided",
}
df["_frame_short"] = df["readability_frame"].map(frame_map).fillna("Undecided")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: stacked horizontal bar by firm
frame_order = ["Li (2008)", "Bushee (2018)", "Undecided"]
frame_palette = {"Li (2008)": "#d62728", "Bushee (2018)": "#2ca02c", "Undecided": "#aaaaaa"}
pivot_frame = df.groupby(["firm", "_frame_short"]).size().unstack(fill_value=0)
pivot_frame = pivot_frame.reindex(firm_order, fill_value=0)
for col_name in frame_order:
    if col_name in pivot_frame.columns:
        pivot_frame[col_name].plot(kind="barh", ax=axes[0], stacked=True,
                                   color=frame_palette[col_name], label=col_name)
axes[0].set_title("Interpretation frame by firm")
axes[0].set_xlabel("Team count")
axes[0].set_ylabel("")
axes[0].legend(title="Frame", bbox_to_anchor=(1.01, 1))

# Right: box plot of Fog scores split by frame
frame_groups = [df[df["_frame_short"] == f]["fog_score"].dropna() for f in frame_order]
axes[1].boxplot(frame_groups, labels=frame_order, patch_artist=True,
                boxprops=dict(facecolor="#e8e8e8"))
axes[1].set_title("Fog score distribution by interpretation frame")
axes[1].set_ylabel("Gunning Fog Index")
axes[1].axhline(19, color="steelblue", linestyle="--", linewidth=0.8,
                label="L&M (2014) lower bound (19)")
axes[1].axhline(21, color="navy",      linestyle=":",  linewidth=0.8,
                label="L&M (2014) upper bound (21)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


### 1.5 TF vs. TF-IDF choice and its effect

> "For teams whose points sit well below the 45-degree line: what does it mean that the TF-IDF score is lower than the raw TF score? What was the TF-IDF weighting removing, and why did its removal reduce the similarity? For teams close to the line: does that mean the two representations agreed, or that boilerplate was a small share of the total vocabulary?"

In [ ]:
#@title Cell 1.5 -- TF vs. TF-IDF cosine similarity scatter

# Each team computed both representations at CP5; every row now has a true
# (cp5_tf_similarity, cp5_tfidf_similarity) coordinate pair.

fig, ax = plt.subplots(figsize=(8, 7))

for _, row in df.iterrows():
    x = pd.to_numeric(row.get("cosine_similarity"), errors="coerce")
    y = pd.to_numeric(row.get("tf_or_tfidf"),    errors="coerce")
    if pd.isna(x) or pd.isna(y):
        continue
    colour = FIRM_COLOURS.get(row["firm"], "grey")
    ax.scatter(x, y, color=colour, s=90, zorder=3)
    ax.annotate(row["team_name"], (x, y),
                textcoords="offset points", xytext=(5, 3), fontsize=8)

# 45-degree parity line (TF == TF-IDF)
lims = [0, 1.02]
ax.plot(lims, lims, "--", color="grey", linewidth=1,
        label="Parity line (TF = TF-IDF)")

# Brown and Tucker (2011) median reference lines
ax.axvline(0.89, color="steelblue", linestyle=":", linewidth=0.9)
ax.axhline(0.89, color="steelblue", linestyle=":", linewidth=0.9,
           label="Brown & Tucker (2011) median (0.89)")

# Quadrant shading
ax.axhspan(0.89, 1.02, xmin=0.89, xmax=1.0, alpha=0.04, color="green")   # both high
ax.axhspan(0.0,  0.89, xmin=0.0,  xmax=0.89, alpha=0.04, color="red")    # both low

# Annotation for below-diagonal region
ax.text(0.3, 0.70, "TF-IDF < TF\n(boilerplate gap)", ha="center", fontsize=8,
        color="grey", style="italic")

# Legend
handles = [mpatches.Patch(color=c, label=f) for f, c in FIRM_COLOURS.items()
           if f in df["firm"].values]
handles += [
    plt.Line2D([0], [0], linestyle="--", color="grey",      label="Parity line"),
    plt.Line2D([0], [0], linestyle=":",  color="steelblue", label="B&T (2011) median"),
]
ax.legend(handles=handles, fontsize=8, bbox_to_anchor=(1.01, 1))
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Cosine similarity -- TF-IDF weighted")
ax.set_ylabel("Cosine similarity -- Raw TF")
ax.set_title("CP5: TF vs TF-IDF cosine similarity (each point = one team)")
plt.tight_layout()
plt.show()

# Summary statistics
tf_col    = pd.to_numeric(df["tf_or_tfidf"],    errors="coerce").dropna()
tfidf_col = pd.to_numeric(df["cosine_similarity"], errors="coerce").dropna()
gap_col   = (tf_col - tfidf_col).dropna()
print(f"\nTF similarity     -- mean: {tf_col.mean():.3f}  std: {tf_col.std():.3f}")
print(f"TF-IDF similarity -- mean: {tfidf_col.mean():.3f}  std: {tfidf_col.std():.3f}")
print(f"Gap (TF - TF-IDF) -- mean: {gap_col.mean():.3f}  max: {gap_col.max():.3f}")
n_below = (tfidf_col < tf_col).sum()
print(f"Teams where TF-IDF < TF (below parity line): {n_below} / {len(tfidf_col)}")


---

## Part 2: Results Analysis

This section analyses the substantive findings across all teams -- sentiment trajectories, readability scores, cosine similarity patterns, and analyst note content. Use these outputs alongside the financial context table to anchor the discussion in observable outcomes.

### 2.1 Sentiment scores by firm and year

> "Expected patterns: Peloton's sentiment should be positive in 2020 and shift meaningfully by 2022. SVB's 2022 score is the one to watch. Boeing's 2019 Risk Factor sentiment should be notably different from 2018. If a firm's sentiment does not move in the expected direction, ask the relevant team whether their pre-processing or dictionary choice might explain it -- before concluding that the method has failed."

In [ ]:
#@title Cell 2.1 -- Sentiment scores by firm and year

# Use L&M net sentiment where available; fall back to Harvard net
df["_sentiment"] = df["net_sentiment_lm"].combine_first(df["net_sentiment_hiv4"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter all teams coloured by firm, year on x-axis
for firm, grp in df.groupby("firm"):
    axes[0].scatter(grp["year"], grp["_sentiment"],
                    color=FIRM_COLOURS.get(firm, "grey"), label=firm, s=90, zorder=3)
    for _, row in grp.iterrows():
        axes[0].annotate(row["team_name"],
                         (row["year"], row["_sentiment"]),
                         textcoords="offset points", xytext=(4, 3), fontsize=7)
axes[0].axhline(0, color="black", linewidth=0.7, linestyle="--")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Net sentiment (L&M)")
axes[0].set_title("Net sentiment by year (all teams)")
axes[0].legend(fontsize=8, bbox_to_anchor=(1.01, 1))

# Right: mean sentiment per firm across years, line plot
firm_year_mean = df.groupby(["firm","year"])["_sentiment"].mean().reset_index()
for firm, grp in firm_year_mean.groupby("firm"):
    grp_sorted = grp.sort_values("year")
    axes[1].plot(grp_sorted["year"], grp_sorted["_sentiment"],
                 marker="o", color=FIRM_COLOURS.get(firm, "grey"), label=firm)
axes[1].axhline(0, color="black", linewidth=0.7, linestyle="--")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Mean net sentiment (L&M)")
axes[1].set_title("Mean net sentiment trajectory by firm")
axes[1].legend(fontsize=8, bbox_to_anchor=(1.01, 1))

plt.tight_layout()
plt.show()


### 2.2 Fog score trajectories

> "Does Fog score increase in the year that net income falls? For which firms is this pattern clearest? For which firms is the Bushee et al. (2018) legitimate complexity argument most plausible? Ask teams who interpreted their score as obfuscation: could an alternative explanation account for the same pattern?"

In [ ]:
#@title Cell 2.2 -- Fog score trajectories and financial context

FINANCIAL_URL = ("https://raw.githubusercontent.com/iamjamesbowden/AG952/main/"
                 "materials/week07/data/financial_context.csv")

try:
    import requests
    fin_df = pd.read_csv(FINANCIAL_URL)
except Exception:
    fin_df = pd.DataFrame(columns=["Firm","Year","NetIncome_M_USD","OperatingMargin_Pct","Note"])
    print("WARNING: could not load financial_context.csv from GitHub.")

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Top: Fog trajectories
fog_mean = df.groupby(["firm","year"])["fog_score"].mean().reset_index()
for firm, grp in fog_mean.groupby("firm"):
    grp_sorted = grp.sort_values("year")
    axes[0].plot(grp_sorted["year"], grp_sorted["fog_score"],
                 marker="o", color=FIRM_COLOURS.get(firm, "grey"), label=firm)
axes[0].axhspan(19, 21, alpha=0.1, color="steelblue",
                label="L&M (2014) benchmark range 19-21")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Gunning Fog Index (mean)")
axes[0].set_title("Fog score trajectories by firm (mean across teams)")
axes[0].legend(fontsize=8, bbox_to_anchor=(1.01, 1))

# Bottom: financial context table
if len(fin_df):
    axes[1].axis("off")
    col_order = ["Firm","Year","NetIncome_M_USD","OperatingMargin_Pct","Note"]
    fin_display = fin_df[col_order].copy()
    fin_display.columns = ["Firm","Year","Net Income ($M)","Op. Margin (%)","Note"]
    table = axes[1].table(
        cellText=fin_display.values,
        colLabels=fin_display.columns,
        cellLoc="left", loc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.auto_set_column_width(col=list(range(len(fin_display.columns))))
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor("#dddddd")
    axes[1].set_title("Financial context", pad=20)

plt.tight_layout()
plt.show()


### 2.3 Cosine similarity heatmap (firm x year-pair)

> "The darkest cells are the most analytically interesting. Boeing 2018-2019 and SVB 2021-2022 should stand out. Ask: for the firms with the lowest similarity scores, was this visible in the language before the event was public knowledge? This is the core question behind Brown and Tucker (2011) -- low similarity is associated with higher information content. Does the pattern here support that?"

In [ ]:
#@title Cell 2.3 -- Cosine similarity heatmap

# Build a firm x year-pair table using mean TF-IDF similarity across teams
heat_rows = []
for _, row in df.iterrows():
    if pd.notna(row.get("comparison_pair")) and pd.notna(row.get("cosine_similarity")):
        heat_rows.append({
            "firm": row["firm"],
            "pair": str(row["comparison_pair"]),
            "similarity": float(row["cosine_similarity"])
        })

if heat_rows:
    heat_df = pd.DataFrame(heat_rows)
    pivot_sim = heat_df.pivot_table(index="firm", columns="pair",
                                    values="similarity", aggfunc="mean")
    pivot_sim = pivot_sim.reindex(firm_order)

    fig, ax = plt.subplots(figsize=(max(8, len(pivot_sim.columns)*1.5), 5))
    sns.heatmap(pivot_sim, annot=True, fmt=".3f", cmap="RdYlGn",
                vmin=0, vmax=1, linewidths=0.5, ax=ax,
                cbar_kws={"label": "Mean TF-IDF cosine similarity"})
    ax.set_title("Mean TF-IDF cosine similarity by firm and year-pair")
    ax.set_xlabel("Year-pair (year t vs. year t-1 or t-2)")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

    # Identify the two lowest-similarity pairs
    flat = heat_df.groupby(["firm","pair"])["similarity"].mean().reset_index()
    flat_sorted = flat.sort_values("similarity")
    print("Lowest mean TF-IDF similarity pairs:")
    print(flat_sorted.head(5).to_string(index=False))
else:
    print("No cosine similarity data available.")


### 2.4 Analyst note vocabulary summary

> "What did the cohort collectively emphasise in their notes? What is absent? If 'limitations' or 'cannot conclude' do not appear in the word cloud, that is informative -- it suggests most notes reported results as conclusions rather than evidence. The examination will reward precision about what the evidence does and does not support."

In [ ]:
#@title Cell 2.4 -- Analyst note word cloud

all_notes = " ".join(df["analyst_note"].dropna().astype(str).tolist()).strip()

if len(all_notes) > 20:
    import re
    stop_extra = {"firm", "year", "the", "a", "an", "and", "or", "is", "was", "are",
                  "were", "of", "in", "to", "that", "this", "our", "we", "for", "with",
                  "from", "at", "by", "as", "it", "its", "on", "be", "has", "have",
                  "been", "not", "which", "their", "than", "also", "more", "no"}

    wc = WordCloud(
        width=900, height=420,
        background_color="white",
        max_words=120,
        stopwords=stop_extra,
        colormap="viridis",
        collocations=True
    ).generate(all_notes)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(wc, interpolation="bilinear")
    axes[0].axis("off")
    axes[0].set_title("Analyst note word cloud (all teams)")

    # Frequency bar chart of top 20 terms
    words = re.findall(r"[a-z]{3,}", all_notes.lower())
    freq = {w: c for w, c in
            sorted([(w, words.count(w)) for w in set(words) if w not in stop_extra],
                   key=lambda x: -x[1])[:20]}
    axes[1].barh(list(freq.keys()), list(freq.values()), color="steelblue")
    axes[1].invert_yaxis()
    axes[1].set_title("Top 20 terms in analyst notes")
    axes[1].set_xlabel("Frequency")

    plt.tight_layout()
    plt.show()

    # Flag absence of hedging language
    hedging = ["limitation", "cannot", "inconclusive", "uncertain", "may", "suggest"]
    found   = [h for h in hedging if h in all_notes.lower()]
    absent  = [h for h in hedging if h not in all_notes.lower()]
    if absent:
        print(f"Hedging terms ABSENT from notes: {absent}")
    if found:
        print(f"Hedging terms present: {found}")
else:
    print("No analyst notes submitted yet.")


### 2.5 Within-firm divergence analysis

> "This is the most direct way to show students that methodological choices have real consequences. If two teams analysed the same Boeing 2018 filing and got sentiment scores that differ by more than 0.05, ask them to trace the difference back through their decision log. Was it the stop word list? The dictionary? The section choice? This is what it means to say that pre-processing choices are not neutral -- the Grimmer, Roberts and Stewart (2022) argument made concrete."

In [ ]:
#@title Cell 2.5 -- Within-firm divergence analysis

# Find teams sharing the same firm AND year, then compare their outcomes
divergence_rows = []
for (firm, year), grp in df.groupby(["firm", "year"]):
    if len(grp) < 2:
        continue
    pairs = []
    grp_list = grp.reset_index(drop=True)
    for i in range(len(grp_list)):
        for j in range(i+1, len(grp_list)):
            a, b = grp_list.iloc[i], grp_list.iloc[j]
            sent_diff = abs((a["_sentiment"] or 0) - (b["_sentiment"] or 0)) \
                        if pd.notna(a["_sentiment"]) and pd.notna(b["_sentiment"]) else None
            fog_diff  = abs((a["fog_score"] or 0) - (b["fog_score"] or 0)) \
                        if pd.notna(a["fog_score"])  and pd.notna(b["fog_score"])  else None
            cos_diff  = abs((a["cosine_similarity"] or 0) - (b["cosine_similarity"] or 0)) \
                        if pd.notna(a.get("cosine_similarity")) and pd.notna(b.get("cosine_similarity")) else None
            divergence_rows.append({
                "Firm": firm, "Year": int(year),
                "Team A": a["team_name"], "Team B": b["team_name"],
                "Norm A": a["normalisation"], "Norm B": b["normalisation"],
                "SW A":   a["stopwords"],    "SW B":   b["stopwords"],
                "Dict A": a["dictionary"],   "Dict B": b["dictionary"],
                "|Delta sentiment|": round(sent_diff, 4) if sent_diff is not None else None,
                "|Delta Fog|":       round(fog_diff,  2) if fog_diff  is not None else None,
                "|Delta TF-IDF cosine|": round(cos_diff, 4) if cos_diff is not None else None,
            })

if divergence_rows:
    div_df = pd.DataFrame(divergence_rows).sort_values(
        "|Delta sentiment|", ascending=False, na_position="last")

    print("Within-firm divergence table:")
    display(div_df.to_string(index=False))

    # Highlight large divergences
    large = div_df[div_df["|Delta sentiment|"].fillna(0) > 0.05]
    if len(large):
        print(f"\n{len(large)} pair(s) with |Delta sentiment| > 0.05 -- "
              f"good candidates for methodological discussion:")
        print(large[["Firm","Year","Team A","Team B","Norm A","Norm B",
                      "SW A","SW B","Dict A","Dict B","|Delta sentiment|"]].to_string(index=False))
    else:
        print("\nNo pairs with |Delta sentiment| > 0.05.")

    # Visualise divergence for the largest
    top5 = div_df.dropna(subset=["|Delta sentiment|"]).head(5)
    if len(top5):
        fig, ax = plt.subplots(figsize=(9, 4))
        labels = [f"{r.Firm} {r.Year}\n{r['Team A']} vs {r['Team B']}" for _, r in top5.iterrows()]
        ax.barh(labels, top5["|Delta sentiment|"],
                color=[FIRM_COLOURS.get(r["Firm"], "grey") for _, r in top5.iterrows()])
        ax.axvline(0.05, color="black", linestyle="--", linewidth=0.8, label="|Delta| = 0.05")
        ax.set_xlabel("|Delta net sentiment|")
        ax.set_title("Top 5 within-firm sentiment divergences")
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print("No teams shared the same firm and year -- divergence analysis not possible.")
